# 🚀 EKVA v2: Routing-as-a-Signal for Sparse MoE KV Cache Compression
### Complete Google Colab Runner (T4 / A100 GPU)

This notebook runs the **EKVA v2** experimental evaluation:
1. **Environment Setup & GPU Verification**
2. **MoE Routing Signature Extraction Hooks** (`Mixtral-8x7B`, `Qwen1.5-MoE-A2.7B`, `DeepSeek-MoE-16B`)
3. **Multi-Signal Token Saliency Engine** ($S(x_t) = w_a \hat{A} + w_r R + w_s \text{Sink} + w_c \text{Recency}$)
4. **Cross-Signal Correlation Check** $\rho(R(x_t), \hat{A}(x_t))$
5. **Multi-Benchmark Evaluation** (GSM8K, HumanEval, PG19 PPL, NIAH Retrieval) vs. Baselines (FullKV, Uniform, H2O, SnapKV, CAKE)
6. **Real Live Model Inference** on Pretrained `Qwen1.5-MoE-A2.7B` on GSM8K prompts
7. **Fused Triton Compaction Kernel & Latency Profiling** (TTFT, TPOT, Speedup)
8. **Publication Plots Generation & Results Export**

In [ ]:
# Step 1: Install Dependencies
!pip install -q torch transformers datasets accelerate triton matplotlib seaborn tqdm pytest bitsandbytes
!nvidia-smi

In [ ]:
# Step 2: Clone or Setup EKVA v2 Repository
import os
if not os.path.exists('EKVA'):
    !git clone https://github.com/GauravPatil2515/EKVA.git
    %cd EKVA
else:
    %cd EKVA
    !git pull

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 3: Run Full Unit Test Suite (CPU + GPU Triton)
!pytest tests/ -v

In [ ]:
# Step 4A: Execute Full EKVA v2 Multi-Benchmark Suite (All 3 Models)
!python3 scripts/run_ekva_v2_experiments.py --out-dir output

In [ ]:
# Step 4B: Run Real Live Inference on Pretrained Qwen1.5-MoE-A2.7B with Real GSM8K Prompts
# Fits easily inside Colab Free Tier T4 GPU (15GB VRAM)!
!python3 scripts/evaluate_real_hf_model.py --model qwen1.5-moe-a2.7b --samples 30 --out-dir output

In [ ]:
# Step 5: Display Publication Figures
from IPython.display import Image, display

if os.path.exists('output/fig2_ablation_curves.png'):
    print("\n📈 Figure 2: Retained Performance Curves across Budgets")
    display(Image('output/fig2_ablation_curves.png'))

if os.path.exists('output/analytical_roofline.png'):
    print("\n📈 Figure 5: Analytical Roofline & Speedup Profile")
    display(Image('output/analytical_roofline.png'))

In [ ]:
# Step 6: Inspect Real Results Summary JSON
import json
if os.path.exists('output/real_eval_qwen1.5-moe-a2.7b.json'):
    with open('output/real_eval_qwen1.5-moe-a2.7b.json') as f:
        real_res = json.load(f)
    print("=" * 60)
    print("REAL QWEN1.5-MoE-A2.7B GSM8K EXACT MATCH SUMMARY")
    print("=" * 60)
    print(json.dumps(real_res, indent=2))

with open('output/ekva_v2_results.json') as f:
    results = json.load(f)

print("=" * 60)
print("EKVA v2 MULTI-BENCHMARK EVALUATION SUMMARY (40% Budget)")
print("=" * 60)
for model, mdata in results.items():
    print(f"\nModel: {model.upper()} (rho = {mdata['correlation_rho']})")
    for task, tdata in mdata['tasks'].items():
        b40 = tdata['40%']
        print(f"  - {task:10s} | EKVA v2: {b40['A+R (EKVA v2)']['mean']} | SnapKV: {b40['SnapKV']['mean']} | H2O: {b40['H2O']['mean']} | CAKE: {b40['CAKE']['mean']}")

In [ ]:
# Step 7: Zip Output Artifacts for Download
!zip -r ekva_v2_colab_artifacts.zip output/
from google.colab import files
files.download('ekva_v2_colab_artifacts.zip')